# Greedy, Intervals, Bits and Math Tricks

Notebook for **Chapter 10** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 The Greedy Idea and the Exchange Argument

In [ ]:
def coins_greedy(amount, coins):
    """Greedy template: order the options, then take what fits."""
    coins = sorted(coins, reverse=True)   # 1. choose the order: biggest first
    used = []                             # 2. the answer we are building
    for c in coins:                       # 3. one pass over the options
        while amount >= c:                # 4. if the option is safe, take it
            amount -= c
            used.append(c)
    return used if amount == 0 else None  # 5. None means "cannot be done"

picked = coins_greedy(63, [1, 5, 10, 25])
print(picked, len(picked))

[25, 25, 10, 1, 1, 1] 6


In [ ]:
def lemonade(bills):
    fives = tens = 0
    for b in bills:
        if b == 5:
            fives += 1
        elif b == 10:
            if fives == 0:
                return False
            fives -= 1
            tens += 1
        else:  # b == 20, change is 15
            if tens > 0 and fives > 0:   # greedy: spend the 10 first
                tens -= 1
                fives -= 1
            elif fives >= 3:
                fives -= 3
            else:
                return False
    return True

assert lemonade([5, 5, 5, 10, 20]) is True
assert lemonade([5, 5, 10, 10, 20]) is False
assert lemonade([10]) is False
print(lemonade([5, 5, 5, 10, 20]), lemonade([5, 5, 10, 10, 20]), lemonade([10]))

True False False


## 10.2 Greedy on Arrays: Cookies, Jump Game, Gas Station

In [ ]:
def find_content_children(g, s):
    g, s = sorted(g), sorted(s)
    child = 0
    for cookie in s:                       # smallest cookie first
        if child < len(g) and cookie >= g[child]:
            child += 1                     # this child is happy
    return child

assert find_content_children([1, 2, 3], [1, 1, 3]) == 2
assert find_content_children([1, 2], [1, 2, 3]) == 2
assert find_content_children([5], [1, 2]) == 0
assert find_content_children([], [1]) == 0
print(find_content_children([1, 2, 3], [1, 1, 3]), find_content_children([1, 2], [1, 2, 3]))

2 2


In [ ]:
def can_jump(nums):
    reach = 0
    for i, jump in enumerate(nums):
        if i > reach:                 # a gap we cannot cross
            return False
        reach = max(reach, i + jump)
    return True

assert can_jump([2, 3, 1, 1, 4]) is True
assert can_jump([3, 2, 1, 0, 4]) is False
assert can_jump([0]) is True
assert can_jump([0, 1]) is False
assert can_jump([2, 0, 0]) is True
print(can_jump([2, 3, 1, 1, 4]), can_jump([3, 2, 1, 0, 4]))

True False


In [ ]:
def min_jumps(nums):
    jumps = end = far = 0
    for i in range(len(nums) - 1):        # never need to jump FROM the last index
        far = max(far, i + nums[i])
        if i == end:                      # wave finished: we must jump now
            jumps += 1
            end = far
    return jumps

assert min_jumps([2, 3, 1, 1, 4]) == 2
assert min_jumps([2, 3, 0, 1, 4]) == 2
assert min_jumps([0]) == 0
assert min_jumps([1, 1, 1, 1]) == 3
assert min_jumps([10, 1, 1]) == 1
print(min_jumps([2, 3, 1, 1, 4]), min_jumps([1, 1, 1, 1]), min_jumps([10, 1, 1]))

2 3 1


In [ ]:
def can_complete_circuit(gas, cost):
    total = tank = start = 0
    for i in range(len(gas)):
        gain = gas[i] - cost[i]
        total += gain
        tank += gain
        if tank < 0:            # stuck: no start in [start, i] can work
            start = i + 1
            tank = 0
    return start if total >= 0 else -1

assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert can_complete_circuit([2, 3, 4], [3, 4, 3]) == -1
assert can_complete_circuit([5], [4]) == 0
assert can_complete_circuit([3, 1, 1], [1, 2, 2]) == 0
print(can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), can_complete_circuit([2, 3, 4], [3, 4, 3]))

3 -1


## 10.3 Interval Scheduling: Activity Selection, Meeting Rooms, Arrows

In [ ]:
def max_non_overlapping(intervals):
    """Interval scheduling template: sort by END, keep what starts after the last kept end."""
    intervals = sorted(intervals, key=lambda iv: iv[1])   # 1. sort by end
    last_end = float("-inf")                              # 2. state: end of the last kept one
    kept = []
    for start, end in intervals:                          # 3. one pass
        if start >= last_end:                             # 4. compatible? touching is allowed
            kept.append((start, end))
            last_end = end
    return kept

acts = [("A", 1, 4), ("B", 3, 5), ("C", 0, 6), ("D", 5, 7), ("E", 3, 9), ("F", 5, 9),
        ("G", 6, 10), ("H", 8, 11), ("I", 8, 12), ("J", 2, 14), ("K", 12, 16)]
by_end = sorted(acts, key=lambda a: a[2])
chosen, last = [], 0
for name, s, e in by_end:
    if s >= last:
        chosen.append(name)
        last = e
print(chosen)
print(len(max_non_overlapping([(s, e) for _, s, e in acts])))

['A', 'D', 'H', 'K']
4


In [ ]:
def can_attend_all(meetings):
    meetings = sorted(meetings)
    for i in range(1, len(meetings)):
        if meetings[i][0] < meetings[i - 1][1]:    # starts before previous ends
            return False
    return True

assert can_attend_all([[7, 10], [2, 4]]) is True
assert can_attend_all([[0, 30], [5, 10], [15, 20]]) is False
assert can_attend_all([[1, 2], [2, 3]]) is True
assert can_attend_all([]) is True
print(can_attend_all([[7, 10], [2, 4]]), can_attend_all([[0, 30], [5, 10], [15, 20]]))

True False


In [ ]:
import heapq

def min_rooms(meetings):
    heap = []                                   # end times of rooms in use
    for start, end in sorted(meetings):
        if heap and heap[0] <= start:           # earliest room is free
            heapq.heapreplace(heap, end)        # pop it and push the new end
        else:
            heapq.heappush(heap, end)           # open a new room
    return len(heap)

assert min_rooms([[0, 30], [5, 10], [15, 20]]) == 2
assert min_rooms([[7, 10], [2, 4]]) == 1
assert min_rooms([[1, 5], [1, 5], [1, 5]]) == 3
assert min_rooms([[1, 2], [2, 3], [3, 4]]) == 1
assert min_rooms([]) == 0
print(min_rooms([[0, 30], [5, 10], [15, 20]]), min_rooms([[1, 5], [1, 5], [1, 5]]))

2 3


In [ ]:
def min_arrows(balloons):
    if not balloons:
        return 0
    balloons = sorted(balloons, key=lambda b: b[1])    # by right edge
    arrows, arrow_x = 1, balloons[0][1]
    for start, end in balloons[1:]:
        if start > arrow_x:                            # missed by the last arrow
            arrows += 1
            arrow_x = end
    return arrows

assert min_arrows([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2
assert min_arrows([[1, 2], [3, 4], [5, 6], [7, 8]]) == 4
assert min_arrows([[1, 2], [2, 3], [3, 4], [4, 5]]) == 2
assert min_arrows([]) == 0
print(min_arrows([[10, 16], [2, 8], [1, 6], [7, 12]]), min_arrows([[1, 2], [2, 3], [3, 4], [4, 5]]))

2 2


In [ ]:
def erase_overlaps(intervals):
    removed, last_end = 0, float("-inf")
    for start, end in sorted(intervals, key=lambda iv: iv[1]):
        if start >= last_end:
            last_end = end
        else:
            removed += 1
    return removed

assert erase_overlaps([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
assert erase_overlaps([[1, 2], [1, 2], [1, 2]]) == 2
assert erase_overlaps([[1, 2], [2, 3]]) == 0
print(erase_overlaps([[1, 2], [2, 3], [3, 4], [1, 3]]), erase_overlaps([[1, 2], [1, 2], [1, 2]]))

1 2


## 10.4 Merging, Inserting and Intersecting Intervals

In [ ]:
def merge(intervals):
    """Merge template: sort by start, extend the last block while the next one overlaps."""
    out = []
    for start, end in sorted(intervals):          # 1. sort by start
        if out and start <= out[-1][1]:           # 2. overlaps the last block?
            out[-1][1] = max(out[-1][1], end)     # 3. extend (use max, not just end)
        else:
            out.append([start, end])              # 4. new block
    return out

print(merge([[1, 3], [2, 6], [8, 10], [15, 18]]))

[[1, 6], [8, 10], [15, 18]]


In [ ]:
def merge_intervals(intervals):
    out = []
    for start, end in sorted(intervals):
        if out and start <= out[-1][1]:
            out[-1][1] = max(out[-1][1], end)
        else:
            out.append([start, end])
    return out

assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert merge_intervals([[1, 4], [0, 5]]) == [[0, 5]]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]
assert merge_intervals([[1, 10], [2, 3], [4, 5]]) == [[1, 10]]
assert merge_intervals([]) == []
print(merge_intervals([[1, 10], [2, 3], [4, 5]]), merge_intervals([[5, 6], [1, 2]]))

[[1, 10]] [[1, 2], [5, 6]]


In [ ]:
def insert_interval(intervals, new):
    out, i, n = [], 0, len(intervals)
    while i < n and intervals[i][1] < new[0]:            # zone 1: strictly before
        out.append(intervals[i])
        i += 1
    while i < n and intervals[i][0] <= new[1]:           # zone 2: overlapping
        new = [min(new[0], intervals[i][0]), max(new[1], intervals[i][1])]
        i += 1
    out.append(new)
    out.extend(intervals[i:])                            # zone 3: strictly after
    return out

assert insert_interval([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]) == [[1, 2], [3, 10], [12, 16]]
assert insert_interval([], [5, 7]) == [[5, 7]]
assert insert_interval([[1, 5]], [6, 8]) == [[1, 5], [6, 8]]
assert insert_interval([[3, 4]], [1, 2]) == [[1, 2], [3, 4]]
print(insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))

[[1, 2], [3, 10], [12, 16]]


In [ ]:
def interval_intersection(A, B):
    i = j = 0
    out = []
    while i < len(A) and j < len(B):
        lo = max(A[i][0], B[j][0])
        hi = min(A[i][1], B[j][1])
        if lo <= hi:
            out.append([lo, hi])
        if A[i][1] < B[j][1]:       # drop the one that ends first
            i += 1
        else:
            j += 1
    return out

A = [[0, 2], [5, 10], [13, 23], [24, 25]]
B = [[1, 5], [8, 12], [15, 24], [25, 26]]
result = interval_intersection(A, B)
assert result == [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]
assert interval_intersection([], B) == []
assert interval_intersection([[1, 3]], [[4, 5]]) == []
print(result)

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


## 10.5 Huffman Coding Awareness and When Greedy Fails

In [ ]:
import heapq

def huffman_codes(freq):
    """Huffman template: merge the two smallest until one tree is left."""
    heap = [(f, i, sym) for i, (sym, f) in enumerate(sorted(freq.items()))]
    heapq.heapify(heap)                       # entries: (frequency, tie-breaker, tree)
    counter = len(heap)
    while len(heap) > 1:
        f1, _, left = heapq.heappop(heap)     # smallest
        f2, _, right = heapq.heappop(heap)    # second smallest
        heapq.heappush(heap, (f1 + f2, counter, (left, right)))
        counter += 1
    codes = {}
    def walk(tree, path):
        if isinstance(tree, str):
            codes[tree] = path or "0"
        else:
            walk(tree[0], path + "0")
            walk(tree[1], path + "1")
    walk(heap[0][2], "")
    return codes

freq = {"a": 45, "b": 13, "c": 12, "d": 16, "e": 9, "f": 5}
codes = huffman_codes(freq)
print(codes)
total_bits = sum(freq[s] * len(c) for s, c in codes.items())
print(total_bits, "bits vs", 3 * sum(freq.values()), "bits with fixed 3-bit codes")

{'a': '0', 'c': '100', 'b': '101', 'f': '1100', 'e': '1101', 'd': '111'}
224 bits vs 300 bits with fixed 3-bit codes


In [ ]:
import heapq

def min_cost_connect(ropes):
    heap = list(ropes)
    heapq.heapify(heap)
    cost = 0
    while len(heap) > 1:
        a = heapq.heappop(heap)
        b = heapq.heappop(heap)
        cost += a + b
        heapq.heappush(heap, a + b)
    return cost

assert min_cost_connect([4, 3, 2, 6]) == 29
assert min_cost_connect([5]) == 0
assert min_cost_connect([]) == 0
assert min_cost_connect([1, 1, 1, 1]) == 8
print(min_cost_connect([4, 3, 2, 6]), min_cost_connect([1, 1, 1, 1]))

29 8


In [ ]:
def greedy_coins(amount, coins):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count if amount == 0 else None

def dp_coins(amount, coins):
    INF = float("inf")
    best = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and best[a - c] + 1 < best[a]:
                best[a] = best[a - c] + 1
    return best[amount] if best[amount] != INF else None

print("greedy", greedy_coins(6, [1, 3, 4]), "dp", dp_coins(6, [1, 3, 4]))
bad = [a for a in range(1, 30) if greedy_coins(a, [1, 3, 4]) != dp_coins(a, [1, 3, 4])]
print("amounts where greedy loses:", bad)
good = [a for a in range(1, 100) if greedy_coins(a, [1, 5, 10, 25]) != dp_coins(a, [1, 5, 10, 25])]
print("US coins, amounts where greedy loses:", good)
assert greedy_coins(6, [1, 3, 4]) == 3 and dp_coins(6, [1, 3, 4]) == 2

greedy 3 dp 2
amounts where greedy loses: [6, 10, 14, 18, 22, 26]
US coins, amounts where greedy loses: []


## 10.6 Bit Manipulation

In [ ]:
def popcount(x):
    """Each loop removes exactly one set bit, so it runs once per 1-bit."""
    count = 0
    while x:
        x &= x - 1        # drop the lowest set bit
        count += 1
    return count

print(popcount(12), popcount(255), popcount(0), bin(12), (12).bit_count())

2 8 0 0b1100 2


In [ ]:
def is_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0

assert is_power_of_two(1) and is_power_of_two(16) and is_power_of_two(1 << 40)
assert not is_power_of_two(0) and not is_power_of_two(12) and not is_power_of_two(-8)
print([n for n in range(0, 40) if is_power_of_two(n)])

[1, 2, 4, 8, 16, 32]


In [ ]:
def single_number(nums):
    acc = 0
    for x in nums:
        acc ^= x
    return acc

assert single_number([4, 1, 2, 1, 2]) == 4
assert single_number([7]) == 7
assert single_number([-3, 5, -3]) == 5
print(single_number([4, 1, 2, 1, 2]), single_number([-3, 5, -3]))

4 5


In [ ]:
def count_bits(n):
    bits = [0] * (n + 1)
    for i in range(1, n + 1):
        bits[i] = bits[i >> 1] + (i & 1)
    return bits

assert count_bits(5) == [0, 1, 1, 2, 1, 2]
assert count_bits(0) == [0]
assert count_bits(16) == [bin(i).count("1") for i in range(17)]
print(count_bits(5), count_bits(8))

[0, 1, 1, 2, 1, 2] [0, 1, 1, 2, 1, 2, 2, 3, 1]


In [ ]:
def subsets(items):
    n = len(items)
    result = []
    for mask in range(1 << n):                       # 0 .. 2^n - 1
        result.append([items[j] for j in range(n) if (mask >> j) & 1])
    return result

def subset_with_sum(nums, target):
    n = len(nums)
    for mask in range(1 << n):
        if sum(nums[j] for j in range(n) if (mask >> j) & 1) == target:
            return [nums[j] for j in range(n) if (mask >> j) & 1]
    return None

assert len(subsets(["a", "b", "c"])) == 8
assert subsets([]) == [[]]
print(subsets(["a", "b", "c"]))
print(subset_with_sum([3, 34, 4, 12, 5, 2], 9))

[[], ['a'], ['b'], ['a', 'b'], ['c'], ['a', 'c'], ['b', 'c'], ['a', 'b', 'c']]
[4, 5]


In [ ]:
def two_singles(nums):
    total = 0
    for x in nums:
        total ^= x                  # equals a ^ b
    low = total & -total            # a bit where a and b differ
    a = b = 0
    for x in nums:
        if x & low:
            a ^= x
        else:
            b ^= x
    return sorted([a, b])

assert two_singles([1, 2, 1, 3, 2, 5]) == [3, 5]
assert two_singles([-1, 0]) == [-1, 0]
assert two_singles([6, 2, 6, 9, 9, 4]) == [2, 4]
print(two_singles([1, 2, 1, 3, 2, 5]), two_singles([6, 2, 6, 9, 9, 4]))

[3, 5] [2, 4]


In [ ]:
def single_number_ii(nums):
    ones = twos = 0
    for x in nums:
        ones = (ones ^ x) & ~twos     # add x to ones unless the bit is already in twos
        twos = (twos ^ x) & ~ones     # add x to twos unless the bit is now in ones
    return ones

assert single_number_ii([2, 2, 3, 2]) == 3
assert single_number_ii([0, 1, 0, 1, 0, 1, 99]) == 99
assert single_number_ii([-2, -2, 1, -2]) == 1
assert single_number_ii([-7, 4, 4, 4]) == -7
print(single_number_ii([2, 2, 3, 2]), single_number_ii([0, 1, 0, 1, 0, 1, 99]))

3 99


## 10.7 Math Tricks: GCD, Primes, Modular Arithmetic, Fast Powers

In [ ]:
import math

def gcd(a, b):
    while b:
        a, b = b, a % b
    return a

def lcm(a, b):
    return a // gcd(a, b) * b        # divide first so the number stays small

print(gcd(48, 18), lcm(4, 6), math.gcd(48, 18), math.lcm(4, 6))

6 12 6 12


In [ ]:
import math

def is_prime(n):
    if n < 2:
        return False
    for d in range(2, math.isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

assert [n for n in range(20) if is_prime(n)] == [2, 3, 5, 7, 11, 13, 17, 19]
assert not is_prime(49) and is_prime(47) and not is_prime(1) and not is_prime(0)
print([n for n in range(30) if is_prime(n)])

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


In [ ]:
import math

def sieve(n):
    """Return a list is_prime[0..n]. Time O(n log log n), space O(n)."""
    is_prime = [True] * (n + 1)
    is_prime[0] = False
    if n >= 1:
        is_prime[1] = False
    for p in range(2, math.isqrt(n) + 1):
        if is_prime[p]:                          # p is a survivor, so it is prime
            for m in range(p * p, n + 1, p):     # smaller multiples were crossed by smaller primes
                is_prime[m] = False
    return is_prime

flags = sieve(49)
print([i for i, ok in enumerate(flags) if ok])
print(sum(sieve(1_000_000)))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
78498


In [ ]:
import math

def count_primes(n):
    if n < 3:
        return 0
    flags = bytearray([1]) * n               # one byte per number (less memory than a list)
    flags[0:2] = b"\x00\x00"
    for p in range(2, math.isqrt(n - 1) + 1):
        if flags[p]:
            flags[p * p::p] = bytearray(len(range(p * p, n, p)))   # zero out the multiples
    return sum(flags)

assert count_primes(0) == 0 and count_primes(2) == 0 and count_primes(3) == 1
assert count_primes(10) == 4
assert count_primes(100) == 25
print(count_primes(10), count_primes(100), count_primes(1_000_000))

4 25 78498


In [ ]:
def power_mod(base, exp, mod):
    """Square-and-multiply. Time O(log exp)."""
    result = 1
    base %= mod
    while exp > 0:
        if exp & 1:                      # this binary digit of the exponent is 1
            result = result * base % mod
        base = base * base % mod         # square: x, x^2, x^4, x^8, ...
        exp >>= 1                        # move to the next binary digit
    return result

print(power_mod(3, 13, 1000), pow(3, 13, 1000), 3 ** 13)

323 323 1594323


In [ ]:
def my_pow(x, n):
    if n < 0:
        x, n = 1 / x, -n
    result = 1.0
    while n:
        if n & 1:
            result *= x
        x *= x
        n >>= 1
    return result

assert my_pow(2.0, 10) == 1024.0
assert my_pow(2.0, -2) == 0.25
assert my_pow(5.0, 0) == 1.0
assert abs(my_pow(1.0001, 10000) - 1.0001 ** 10000) < 1e-9
print(my_pow(2.0, 10), my_pow(2.0, -2), my_pow(5.0, 0))

1024.0 0.25 1.0


In [ ]:
import math

P = 1_000_000_007
N = 2000
fact = [1] * (N + 1)
for i in range(1, N + 1):
    fact[i] = fact[i - 1] * i % P

def nCr_mod(n, k):
    if k < 0 or k > n:
        return 0
    inv_k = pow(fact[k], P - 2, P)          # inverse of k! by Fermat's little theorem
    inv_nk = pow(fact[n - k], P - 2, P)     # inverse of (n-k)!
    return fact[n] * inv_k % P * inv_nk % P

assert nCr_mod(10, 3) == 120
assert nCr_mod(5, 0) == 1 and nCr_mod(5, 5) == 1 and nCr_mod(3, 5) == 0
assert nCr_mod(1000, 500) == math.comb(1000, 500) % P
assert pow(fact[7], -1, P) == pow(fact[7], P - 2, P)
print(nCr_mod(10, 3), nCr_mod(1000, 500))

120 159835829


## 10.8 Randomness: Reservoir Sampling and Fair Shuffling

In [ ]:
import random

def reservoir_one(stream, rng):
    """After i items, each of them is the chosen one with probability 1/i."""
    chosen = None
    for i, item in enumerate(stream, start=1):
        if rng.randrange(i) == 0:       # true with probability 1/i
            chosen = item
    return chosen

rng = random.Random(7)
counts = {x: 0 for x in range(5)}
for _ in range(10000):
    counts[reservoir_one(range(5), rng)] += 1
print(all(1700 <= c <= 2300 for c in counts.values()), sum(counts.values()))

True 10000


In [ ]:
from fractions import Fraction

def final_probability(i, n):
    p = Fraction(1, i)                      # picked at step i
    for j in range(i + 1, n + 1):
        p *= 1 - Fraction(1, j)             # survives step j
    return p

probs = [final_probability(i, 5) for i in range(1, 6)]
print([str(p) for p in probs])
assert all(p == Fraction(1, 5) for p in probs)

['1/5', '1/5', '1/5', '1/5', '1/5']


In [ ]:
import random

def reservoir_k(stream, k, rng):
    sample = []
    for i, item in enumerate(stream, start=1):
        if i <= k:
            sample.append(item)
        else:
            j = rng.randrange(i)             # 0 .. i-1
            if j < k:
                sample[j] = item
    return sample

rng = random.Random(2024)
hits = [0] * 10
trials = 20000
for _ in range(trials):
    for x in reservoir_k(range(10), 3, rng):
        hits[x] += 1
expected = trials * 3 / 10                   # k / n of the trials
print(all(abs(h - expected) < 0.05 * expected for h in hits))
assert reservoir_k([1, 2], 5, rng) == [1, 2]
assert reservoir_k([], 3, rng) == []
print(sum(hits) == trials * 3)

True
True


In [ ]:
from itertools import product
from collections import Counter

def naive(choices):
    a = [0, 1, 2]
    for i, j in enumerate(choices):          # j may be ANY position 0..2
        a[i], a[j] = a[j], a[i]
    return tuple(a)

def fisher_yates(choices):
    a = [0, 1, 2]
    for i, j in zip((2, 1), choices):        # j only in 0..i
        a[i], a[j] = a[j], a[i]
    return tuple(a)

naive_counts = Counter(naive(c) for c in product(range(3), repeat=3))
fy_counts = Counter(fisher_yates(c) for c in product(range(3), range(2)))
print(sorted(naive_counts.items()))
print(sorted(fy_counts.items()))
assert sum(naive_counts.values()) == 27 and len(fy_counts) == 6
assert all(v == 1 for v in fy_counts.values())

[((0, 1, 2), 4), ((0, 2, 1), 5), ((1, 0, 2), 5), ((1, 2, 0), 5), ((2, 0, 1), 4), ((2, 1, 0), 4)]
[((0, 1, 2), 1), ((0, 2, 1), 1), ((1, 0, 2), 1), ((1, 2, 0), 1), ((2, 0, 1), 1), ((2, 1, 0), 1)]


## Chapter recap

## Practice set

Twenty-four problems, graded from easy to hard. Try each one on paper before reading the answer.

In [ ]:
print(13 & 6, 13 | 6, 13 ^ 6, 13 << 2, 13 >> 2, ~13)

In [ ]:
def merge_bad(intervals):
    out = []
    for start, end in sorted(intervals):
        if out and start <= out[-1][1]:
            out[-1][1] = end              # bug is here
        else:
            out.append([start, end])
    return out

print(merge_bad([[1, 10], [2, 3], [4, 5]]))

In [ ]:
import heapq

def rooms_bad(meetings):
    heap = []
    for start, end in sorted(meetings):
        if heap and heap[0] < start:        # bug is here
            heapq.heapreplace(heap, end)
        else:
            heapq.heappush(heap, end)
    return len(heap)

print(rooms_bad([[1, 2], [2, 3]]))

In [ ]:
def arrows_bad(balloons):
    balloons = sorted(balloons)               # sorted by start: bug
    arrows, x = 1, balloons[0][1]
    for s, e in balloons[1:]:
        if s > x:
            arrows += 1
            x = e
    return arrows

print(arrows_bad([[1, 6], [2, 3], [4, 5]]))

In [ ]:
import math

def trailing_zeros_factorial(n):
    count, p = 0, 5
    while p <= n:
        count += n // p
        p *= 5
    return count

s = str(math.factorial(100))
assert trailing_zeros_factorial(100) == len(s) - len(s.rstrip("0"))
print(trailing_zeros_factorial(100), trailing_zeros_factorial(25), trailing_zeros_factorial(4))

24 6 0


In [ ]:
print(pow(7, 222, 10), [pow(7, e, 10) for e in range(1, 9)])

9 [7, 9, 3, 1, 7, 9, 3, 1]


In [ ]:
print(-7 % 3, 7 % -3, -7 // 2, divmod(-7, 3))

In [ ]:
def sieve_bad(n):
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(n ** 0.5) + 1):
        if is_prime[p]:
            for m in range(p, n + 1, p):      # bug is here
                is_prime[m] = False
    return [i for i, ok in enumerate(is_prime) if ok]

print(sieve_bad(20))

In [ ]:
import heapq

freqs = [1, 1, 2, 4]
heap = list(freqs)
heapq.heapify(heap)
cost = 0
while len(heap) > 1:
    a, b = heapq.heappop(heap), heapq.heappop(heap)
    cost += a + b
    heapq.heappush(heap, a + b)
assert cost == 14
print(cost)

14


In [ ]:
def missing_number(nums):
    acc = len(nums)                  # the index n has no slot, so start with it
    for i, x in enumerate(nums):
        acc ^= i ^ x
    return acc

assert missing_number([3, 0, 1]) == 2
assert missing_number([0, 1]) == 2
assert missing_number([9, 6, 4, 2, 3, 5, 7, 0, 1]) == 8
assert missing_number([0]) == 1
print(missing_number([3, 0, 1]), missing_number([0, 1]))

2 2


In [ ]:
from itertools import product
from collections import Counter

def naive(choices):
    a = [0, 1, 2]
    for i, j in enumerate(choices):
        a[i], a[j] = a[j], a[i]
    return tuple(a)

counts = Counter(naive(c) for c in product(range(3), repeat=3))
print(sorted(counts.values()), sum(counts.values()))

In [ ]:
def max_overlap(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))
        events.append((e, -1))
    events.sort()                      # at the same time, -1 comes before +1
    active = best = 0
    for _, delta in events:
        active += delta
        best = max(best, active)
    return best

assert max_overlap([[1, 4], [2, 5], [3, 6], [7, 8]]) == 3
assert max_overlap([[1, 2], [2, 3]]) == 1
assert max_overlap([]) == 0
print(max_overlap([[1, 4], [2, 5], [3, 6], [7, 8]]), max_overlap([[1, 2], [2, 3]]))

3 1


In [ ]:
def is_power_of_four(n):
    return n > 0 and (n & (n - 1)) == 0 and (n & 0x55555555) != 0

assert [n for n in range(1, 70) if is_power_of_four(n)] == [1, 4, 16, 64]
print([n for n in range(1, 300) if is_power_of_four(n)])

[1, 4, 16, 64, 256]


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch10.html